In [ ]:
from sklearn.neighbors import NearestNeighbors
import glob
import pandas as pd
from astropy.cosmology import Planck18
from astropy import units as u
import numpy as np
from astropy.coordinates import SkyCoord
from scipy.stats import norm
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

In [ ]:
barred_target = pd.read_csv('../data/matched_barred_ps1_resolved.csv')
unbarred_target = pd.read_csv('../data/matched_unbarred_ps1_resolved.csv')

In [ ]:
target = pd.read_csv('../all_sample_20260702.csv')
target = pd.merge(target, barred_target[['paliya_RA', 'class']], how='left', on='paliya_RA')
target = pd.merge(target, unbarred_target[['paliya_RA', 'class']], how='left', on='paliya_RA')
target['class'] = target['class_x'].combine_first(target['class_y'])
target = target.drop(['class_x', 'class_y'], axis=1)

target = target[target['class'].isin(['barred', 'unbarred'])]
tg_z = (target.paliya_Z).values
tg_d = Planck18.comoving_distance(target.paliya_Z).value
tg_ra = (target.paliya_RA).values
tg_dec = (target.paliya_DEC).values

coord = SkyCoord(ra = tg_ra * u.deg, dec = tg_dec * u.deg, distance = tg_d * u.Mpc)
tg_df = pd.DataFrame({'coord':coord})

In [ ]:
from astropy.coordinates import Distance
from astropy.cosmology import z_at_value


def compute_index_for_target(i):
    file_list = glob.glob(f'../data/catalog/ps1/redshift_STRM/t{tg_ra[i]:08.4f}{tg_dec[i]:+07.4f}.csv')
    if len(file_list) == 0:
        return -999

    target_z = float(target.paliya_Z.iloc[i])
    df = pd.read_csv(file_list[0])

    z_window = (
        z_at_value(Planck18.comoving_distance, (tg_d[i] + 5) * u.Mpc)
        - z_at_value(Planck18.comoving_distance, (tg_d[i] - 5) * u.Mpc)
    )
    # df = df[np.isfinite(df.z_phot0) & np.isfinite(df.z_photErr)]
    # df = df[np.abs(df.z_phot0 - target_z) < z_window]

    if len(df) == 0:
        return -999

    surround_z = df['z_phot0'].to_numpy(dtype=float)
    surround_err = df['z_photErr'].to_numpy(dtype=float)
    surround_ra = df.raMean.to_numpy(dtype=float)
    surround_dec = df.decMean.to_numpy(dtype=float)

    mask = (surround_z - surround_err / 2) > -1
    surround_z = surround_z[mask]
    surround_err = surround_err[mask]
    surround_ra = surround_ra[mask]
    surround_dec = surround_dec[mask]

    if len(surround_z) == 0:
        return -999

    surround_d = Planck18.comoving_distance(surround_z).value
    valid = np.isfinite(surround_d) & (surround_d >= 0)
    surround_z = surround_z[valid]
    surround_err = surround_err[valid]
    surround_ra = surround_ra[valid]
    surround_dec = surround_dec[valid]
    surround_d = surround_d[valid]

    if len(surround_d) == 0:
        return -999

    z_low = np.maximum(surround_z - surround_err / 2, 0)
    surround_scale = (
        Planck18.comoving_distance(surround_z + surround_err / 2)
        - Planck18.comoving_distance(z_low)
    ).value
    surround_odds = np.abs(
        norm.cdf(Planck18.comoving_distance(tg_z[i] + 1000/300000), loc=surround_d, scale=surround_scale)
        - norm.cdf(Planck18.comoving_distance(tg_z[i] - 1000/300000), loc=surround_d, scale=surround_scale)
    )

    coord_1 = SkyCoord(ra=surround_ra * u.deg, dec=surround_dec * u.deg, distance=surround_d * u.Mpc)
    target_coord = coord[i]

    df_surround = target_coord.separation(coord_1)
    df_surround = df_surround.to_value(u.rad) * tg_d[i]
    X = np.asarray(df_surround, dtype=float).reshape(-1, 1)
    nn = NearestNeighbors(n_neighbors=max(1, min(int(len(X) / 2), len(X))), algorithm='auto')
    nn.fit(X)
    neighbors = nn.kneighbors(np.array([[0]]))
    k_index = np.where(np.cumsum(surround_odds[neighbors[1][0]]) >= 5)[0]
    #calculate the density

    if len(k_index) == 0:
        return -999
    index = neighbors[1][0][:k_index[0]]
    
    row = df.distance_arcmin.iloc[index] / 60
    row = np.deg2rad(row)
    fifth_dis = row * tg_d[i]
    fifth_dis = (fifth_dis**2) * surround_odds[index]
    fifth_dens = np.sum(surround_odds[index]) / np.pi / np.sum(fifth_dis)
    fifth_dens = np.log(fifth_dens)

    return fifth_dens

dens = np.array(Parallel(n_jobs=-1, verbose=5)(delayed(compute_index_for_target)(i) for i in range(len(tg_df))))

In [ ]:
dens = np.array(dens)[dens != -999]

In [ ]:
np.save('./STRM_dens.npy', dens)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dens, bins=20, label='Density Distribution', color='orange', alpha=0.7, cumulative=False)
plt.xlabel('Density (5th Nearest Neighbor)')
plt.ylabel('Frequency')
plt.title('Histogram of Density Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('../fig/density_distribution_histogram_STRM.png', dpi=300)
plt.show()

In [ ]:
# index_arr[index_arr != -999].size

In [ ]:
# dens = []
# for i, idx in enumerate(index_arr):
#     if idx == -999:
#         continue
#     idx = int(idx)
#     file_list = glob.glob(f'../data/catalog/ps1/redshift_STRM/t{tg_ra[i]:08.4f}{tg_dec[i]:+07.4f}.csv')
#     if len(file_list) == 0:
#         continue
#     df = pd.read_csv(file_list[0])
#     if idx >= len(df):
#         continue
#     row = df.distance_arcmin.iloc[idx] / 60
#     row = np.deg2rad(row)
#     fifth_dis = row * tg_d[i]
#     if fifth_dis <= 0 or not np.isfinite(fifth_dis):
#         continue
#     fifth_dens = 5 / np.pi / fifth_dis**2
#     fifth_dens = np.log(fifth_dens)
#     dens.append(fifth_dens)


In [ ]:
# plt.hist(dens, bins = 50, label = 'Density Distribution', color = 'blue', alpha = 0.7)
# plt.xlabel('Density (5th Nearest Neighbor)')
# plt.ylabel('Frequency')
# plt.title('Histogram of Density Distribution')
# plt.legend()
# plt.savefig('../fig/density_distribution_histogram_STRM.png', dpi=300)
# plt.show()